# Circuit Simplification - Local Execution

This notebook demonstrates running circuit simplification locally using `obi-one`.

In production, jobs are launched via TaskManager. This notebook lets you run and debug simplification locally before submitting to the task queue.

**Two modes:**
1. **From entitycore ID** - Fetches and stages the circuit from entitycore (production workflow)
2. **From local path** - Uses a local circuit file (for testing)

**Prerequisites:**
- `obi-one` installed
- `sonata_simplify` installed  
- NEURON with compiled mod files (for `single_compartment` algorithm)

**Writes to:** `obi-output/circuit_simplification/` containing:
- `obi_one_coordinate.json` - the TaskConfig
- `simplified/circuit_config.json` - the simplified circuit

## Imports

In [1]:
import json
from pathlib import Path

import obi_one as obi
from entitysdk import Client, ProjectContext
from obi_auth import get_token
from obi_one.core.info import Info
from obi_one.db_sdk import db_sdk
from obi_one.scientific.from_id.circuit_from_id import CircuitFromID
from obi_one.scientific.library.circuit import Circuit
from obi_one.scientific.tasks.circuit_simplification import (
    CircuitSimplificationScanConfig,
    CircuitSimplificationTask,
)

## Connect to entitycore staging

In [2]:
virtual_lab_id = obi.LAB_ID_STAGING_TEST
project_id = obi.PROJECT_ID_STAGING_TEST

token = get_token(environment="staging")
project_context = ProjectContext(virtual_lab_id=virtual_lab_id, project_id=project_id)
db_client = Client(
    api_url="https://staging.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)
print("Connected to entitycore staging.")

Connected to entitycore staging.


## Configuration

Choose one of the two modes:

**Mode 1: From entitycore ID** (production workflow)  
Set `circuit_id` to the UUID of a circuit in entitycore.

**Mode 2: From local path** (for testing)  
Set `circuit_id = None` and provide a local `circuit_path`.

In [3]:
# ============================================================
# MODE 1: From entitycore ID (set circuit_id to a UUID string)
# ============================================================
circuit_id = "4733054a-edc5-46cb-80cd-748758879355"  # nbS1-O1-sSub-pre-dim5-nCN-HEX0-L6-01

# ============================================================
# MODE 2: From local path (set circuit_id = None)
# ============================================================
# circuit_id = None
# local_circuit_path = Path("../../../../data/tiny_circuits/N_10__top_nodes_dim6/circuit_config.json")

# Output directory
output_root = Path("../../../../../../obi-output/circuit_simplification")
output_root = output_root.resolve()
output_root.mkdir(parents=True, exist_ok=True)

print(f"Output root: {output_root}")

Output root: /Users/mandge/Desktop/obi/inait/obi-output/circuit_simplification


## Clean Up Old Outputs

Remove previous simplification runs to avoid stale results interfering with the new run.  
Set `clean_output = True` to delete all contents of the output root (except the entity cache).

In [4]:
import shutil

clean_output = True  # Set to False to keep old runs

if clean_output:
    entity_cache = output_root / "entity_cache"
    for item in output_root.iterdir():
        if item == entity_cache:
            continue
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()
    print(f"Cleaned output root (kept entity_cache): {output_root}")
else:
    print(f"Keeping existing outputs in: {output_root}")

Cleaned output root (kept entity_cache): /Users/mandge/Desktop/obi/inait/obi-output/circuit_simplification


## Create Circuit Reference

If using a circuit ID, this creates a `CircuitFromID` reference.  
If using a local path, this creates a `Circuit` object directly.

In [5]:
if circuit_id is not None:
    # Mode 1: From entitycore ID
    circuit_ref = CircuitFromID(id_str=circuit_id)
    
    # Fetch entity info to display
    circuit_entity = circuit_ref.entity(db_client=db_client)
    print(f"Circuit from entitycore:")
    print(f"  ID: {circuit_id}")
    print(f"  Name: {circuit_entity.name}")
    print(f"  Description: {circuit_entity.description[:200]}..." if len(circuit_entity.description or '') > 200 else f"  Description: {circuit_entity.description}")
else:
    # Mode 2: From local path
    circuit_path = local_circuit_path.resolve()
    if not circuit_path.exists():
        raise FileNotFoundError(f"Circuit config not found: {circuit_path}")
    
    circuit_ref = Circuit(name="local_circuit", path=str(circuit_path))
    print(f"Local circuit: {circuit_path}")

Circuit from entitycore:
  ID: 4733054a-edc5-46cb-80cd-748758879355
  Name: nbS1-O1-sSub-pre-dim5-nCN-HEX0-L6-01
  Description: A simplicial subcircuit (sSub) extracted from the nbS1-O1 circuit, around a neuron with high centrality in the network’s connectivity graph, located in layer 6 of subcolumn HEX0. The subnetwork consis...


## Stage Circuit (for entitycore circuits)

When using a circuit ID, we need to download (stage) the circuit from entitycore.  
This step downloads the circuit assets to a local cache directory.

**Note:** This may take a while for large circuits. If it fails with a network error, retry this cell.

In [6]:
import tempfile

if circuit_id is not None:
    print(f"Staging circuit from entitycore...")
    print(f"  Circuit ID: {circuit_id}")
    print("  (This may take a while for large circuits)")
    print()

    # Stage into a persistent cache directory so retries don't re-download.
    # temp_dir is required by the API even when entity_cache=True.
    with tempfile.TemporaryDirectory() as tmp:
        circuit, circuit_entity = db_sdk.resolve_circuit(
            circuit_ref,
            db_client=db_client,
            entity_cache=True,
            cache_root=output_root,
            temp_dir=Path(tmp),
        )

    print(f"Circuit staged successfully!")
    print(f"  Local path: {circuit.path}")
else:
    # Local circuit - no staging needed
    circuit = circuit_ref
    circuit_entity = None
    print("Using local circuit - no staging needed.")


Staging circuit from entitycore...
  Circuit ID: 4733054a-edc5-46cb-80cd-748758879355
  (This may take a while for large circuits)

Circuit staged successfully!
  Local path: /Users/mandge/Desktop/obi/inait/obi-output/circuit_simplification/entity_cache/sonata_circuit/4733054a-edc5-46cb-80cd-748758879355/circuit_config.json


## Compile NEURON Mechanisms (for `single_compartment` algorithm)

The `single_compartment` algorithm requires NEURON with the circuit's MOD files compiled.  
This step runs `nrnivmodl` on the staged circuit's `mod/` directory.

**Skip this cell if:** using only point-neuron algorithms (`lif`, `adex`, `izhikevich`, `glif`, `gif`) without `single_compartment`.

In [7]:
import platform
import subprocess
import sys

circuit_dir = Path(circuit.path).parent
mod_dir = circuit_dir / "mod"
arch = "arm64" if platform.machine() == "arm64" else "x86_64"
compiled_dir = circuit_dir / arch

if mod_dir.exists():
    # Check for empty .mod files (data corruption from staging)
    empty_mods = [f for f in mod_dir.glob("*.mod") if f.stat().st_size == 0]
    if empty_mods:
        print(f"Found {len(empty_mods)} empty MOD file(s):")
        for f in empty_mods:
            print(f"  - {f.name}")
        # Try to find valid copies from the tiny_circuits examples
        fallback_dirs = [
            Path("../../data/tiny_circuits/N_10__top_nodes_dim6/mod"),
            Path("../../data/tiny_circuits/nbS1-O1-E2Sst-maxNsyn-HEX0-L5/mod"),
        ]
        for empty_mod in empty_mods:
            found = False
            for fallback in fallback_dirs:
                candidate = fallback.resolve() / empty_mod.name
                if candidate.exists() and candidate.stat().st_size > 0:
                    import shutil
                    shutil.copy2(candidate, empty_mod)
                    print(f"  Replaced {empty_mod.name} from {candidate}")
                    found = True
                    break
            if not found:
                print(f"  WARNING: No valid copy found for {empty_mod.name}, removing")
                empty_mod.unlink()

    if not compiled_dir.exists() or not (compiled_dir / "libnrnmech.dylib").exists():
        print(f"Compiling MOD files in {mod_dir}...")
        nrnivmodl = Path(sys.prefix) / "bin" / "nrnivmodl"
        if not nrnivmodl.exists():
            nrnivmodl = "nrnivmodl"
        result = subprocess.run(
            [str(nrnivmodl), "-incflags", "-DDISABLE_REPORTINGLIB", str(mod_dir)],
            cwd=circuit_dir,
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            print(f"nrnivmodl failed (exit {result.returncode}):")
            print(result.stderr[-2000:] if result.stderr else "(no stderr)")
        else:
            print(f"MOD files compiled successfully: {compiled_dir}")
    else:
        print(f"Mechanisms already compiled: {compiled_dir}")
else:
    print(f"No mod/ directory found at {mod_dir}")
    print("Skipping compilation (may not be needed for point-neuron algorithms)")

Mechanisms already compiled: /Users/mandge/Desktop/obi/inait/obi-output/circuit_simplification/entity_cache/sonata_circuit/4733054a-edc5-46cb-80cd-748758879355/arm64


## Inspect Circuit

In [8]:
# Inspect the circuit
sonata = circuit.sonata_circuit
print(f"Circuit: {circuit.name}")
print(f"Path: {circuit.path}")
print(f"\nNode populations:")
for pop_name in sonata.nodes.population_names:
    pop = sonata.nodes[pop_name]
    print(f"  - {pop_name}: {pop.size} neurons")

print(f"\nEdge populations:")
for pop_name in sonata.edges.population_names:
    pop = sonata.edges[pop_name]
    print(f"  - {pop_name}: {pop.size} synapses")

Circuit: CircuitFromID_4733054a-edc5-46cb-80cd-748758879355
Path: /Users/mandge/Desktop/obi/inait/obi-output/circuit_simplification/entity_cache/sonata_circuit/4733054a-edc5-46cb-80cd-748758879355/circuit_config.json

Node populations:
  - POm: 140 neurons
  - S1nonbarrel_neurons: 6 neurons
  - VPM: 116 neurons
  - external_S1nonbarrel_neurons: 3233 neurons

Edge populations:
  - POm__S1nonbarrel_neurons__chemical: 229 synapses
  - S1nonbarrel_neurons__S1nonbarrel_neurons__chemical: 112 synapses
  - VPM__S1nonbarrel_neurons__chemical: 208 synapses
  - external_S1nonbarrel_neurons__S1nonbarrel_neurons__chemical: 13947 synapses


In [9]:
# Show etype distribution (important for simplification parameters)
for pop_name in sonata.nodes.population_names:
    pop = sonata.nodes[pop_name]
    try:
        df = pop.get()
        if "etype" in df.columns:
            print(f"\nEtype distribution for {pop_name}:")
            print(df["etype"].value_counts())
    except Exception as e:
        print(f"Could not get etype distribution for {pop_name}: {e}")


Etype distribution for S1nonbarrel_neurons:
etype
cADpyr    6
Name: count, dtype: int64

Etype distribution for external_S1nonbarrel_neurons:
etype
cADpyr    3094
cACint      40
cNAC        22
bIR         19
bAC         17
bSTUT       14
cSTUT       13
dNAC         5
dSTUT        5
bNAC         2
cIR          2
Name: count, dtype: int64


## Build the Scan Config

Available algorithms:
- `single_compartment`: Rossert point-neuron (SONATA/NEURON only)
- `lif`: Leaky integrate-and-fire (SONATA + NEST export)
- `adex`: Adaptive exponential I&F (SONATA + NEST + Brian2 export)
- `izhikevich`: Izhikevich model (SONATA only)
- `glif`: Generalized LIF (SONATA + NEST export)
- `gif`: Generalized I&F (SONATA only)

Each algorithm always produces a SONATA circuit (registered with `target_simulator=NEURON`).  
Algorithms with NEST/Brian2 exports additionally produce separate Circuit entities with the appropriate `target_simulator`.

In [10]:
# Use the staged circuit (local Circuit object) for the scan config
# This avoids re-staging during task execution
scan_config = CircuitSimplificationScanConfig(
    info=Info(
        campaign_name="Circuit Simplification Test",
        campaign_description="Testing circuit simplification locally",
    ),
    initialize=CircuitSimplificationScanConfig.Initialize(
        circuit=circuit,  # Use the staged Circuit, not CircuitFromID
    ),
    simplification=CircuitSimplificationScanConfig.Simplification(
        algorithms=["single_compartment"],
    ),
)

print("Scan config created.")
print(f"  Circuit: {circuit.name}")
print(f"  Algorithms: {scan_config.simplification.algorithms}")

Scan config created.
  Circuit: CircuitFromID_4733054a-edc5-46cb-80cd-748758879355
  Algorithms: ['single_compartment']


## Generate TaskConfig

In [11]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root=str(output_root),
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print(f"Coordinate output root: {coord_root}")
print(f"Generated {len(grid_scan.single_configs)} task config(s)")

Coordinate output root: /Users/mandge/Desktop/obi/inait/obi-output/circuit_simplification/0
Generated 1 task config(s)


/Users/mandge/Desktop/obi/inait/obi-one/.venv/lib/python3.12/site-packages/pydantic/main.py:542: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `list[literal['single_compartment','lif','adex','izhikevich','glif','gif']]` - serialized value may not be as expected [field_name='algorithms', input_value='single_compartment', input_type=str])
  return self.__pydantic_serializer__.to_json(


## Run the Simplification Task

This is the main execution step. The task will:
1. Run the simplification pipeline on the staged circuit
2. Optionally register the output circuit entity in entitycore

**Note:** The simplification can take several minutes depending on circuit size.

In [12]:
single_config = grid_scan.single_configs[0]
task = CircuitSimplificationTask(config=single_config)

print("Running simplification task...")
print("(This may take several minutes for filter computation)")

# Pass db_client to enable registering output circuits to entitycore
# Set db_client=None for local-only execution without registration
result = task.execute(db_client=db_client)

print(f"\nSimplification complete!")
if result:
    print(f"Registered circuit entity ID: {result}")
else:
    print("No circuit was registered (local-only execution or registration disabled)")

Running simplification task...
(This may take several minutes for filter computation)


Processing edge populations:   0%|          | 0/4 [00:00<?, ?pop/s, S1nonbarrel_neurons__S1nonbarrel_neurons__chemical]No sane fit achieved for N=1, RMSE: 0.07818027877863641
No sane fit achieved for N=1, RMSE: 0.05238531400075472
No sane fit achieved for N=1, RMSE: 0.05137250612123224
No sane fit achieved for N=1, RMSE: 0.05074693182127306
No sane fit achieved for N=1, RMSE: 0.10332749203777636
No sane fit achieved for N=1, RMSE: 0.05917435398079157
No sane fit achieved for N=1, RMSE: 0.06339076936711173
No sane fit achieved for N=1, RMSE: 0.08609039331871185
No sane fit achieved for N=1, RMSE: 0.07726201353900367
No sane fit achieved for N=1, RMSE: 0.1009610903562095
No sane fit achieved for N=1, RMSE: 0.10459095008067339
No sane fit achieved for N=1, RMSE: 0.06478128387894778
No sane fit achieved for N=1, RMSE: 0.08709971597951555
No sane fit achieved for N=1, RMSE: 0.10501562307561461
No sane fit achieved for N=1, RMSE: 0.05486309774841001
No sane fit achieved for N=1, RMSE: 0.1036


Simplification complete!
No circuit was registered (local-only execution or registration disabled)


## Inspect Results

In [13]:
# Find generated simplified circuits
simplified_configs = list(output_root.glob("**/simplified/circuit_config.json"))
print(f"Found {len(simplified_configs)} simplified circuit(s):")
for cfg in simplified_configs:
    print(f"  - {cfg}")

Found 1 simplified circuit(s):
  - /Users/mandge/Desktop/obi/inait/obi-output/circuit_simplification/0/single_compartment/simplified/circuit_config.json


In [14]:
# Load and compare original vs simplified
from bluepysnap import Circuit as SnapCircuit

if simplified_configs:
    simplified_circuit = SnapCircuit(str(simplified_configs[0]))
    
    print("=== Original Circuit ===")
    for pop_name in sonata.nodes.population_names:
        print(f"{pop_name}: {sonata.nodes[pop_name].size} neurons")
    
    print("\n=== Simplified Circuit ===")
    for pop_name in simplified_circuit.nodes.population_names:
        print(f"{pop_name}: {simplified_circuit.nodes[pop_name].size} neurons")
else:
    print("No simplified circuits found. Check logs for errors.")

=== Original Circuit ===
POm: 140 neurons
S1nonbarrel_neurons: 6 neurons
VPM: 116 neurons
external_S1nonbarrel_neurons: 3233 neurons

=== Simplified Circuit ===
POm: 140 neurons
S1nonbarrel_neurons: 6 neurons
VPM: 116 neurons
external_S1nonbarrel_neurons: 3233 neurons


In [15]:
# Show simplified neuron properties
if simplified_configs:
    for pop_name in simplified_circuit.nodes.population_names:
        pop = simplified_circuit.nodes[pop_name]
        df = pop.get()
        print(f"\n=== {pop_name} properties ===")
        # Show relevant columns for simplified neurons
        display_cols = [c for c in df.columns if c in [
            'etype', 'mtype', 'model_type', 'model_template',
            'x', 'y', 'z', '@dynamics:holding_current'
        ]]
        if display_cols:
            print(df[display_cols].head(10))
        else:
            print(df.head(10))


=== POm properties ===
         model_template model_type            x            y            z
node_ids                                                                 
0                          virtual  3176.474713 -1061.091809 -3468.392582
1                          virtual  3243.487253  -767.672022 -3467.687838
2                          virtual  3167.636921  -974.124921 -3452.391431
3                          virtual  3229.875975  -604.895717 -3410.225060
4                          virtual  3195.840449  -640.101235 -3397.215168
5                          virtual  3219.465334  -966.948614 -3489.520491
6                          virtual  3232.597729  -819.604375 -3471.493468
7                          virtual  3193.894084 -1145.952364 -3486.294241
8                          virtual  3260.042528  -582.796656 -3424.467103
9                          virtual  3289.165560  -609.362653 -3453.873808

=== S1nonbarrel_neurons properties ===
           etype model_template   model_type    

---

## Notes

**Bundled Parameters:**  
The `sonata_simplify` package includes default simplification parameters for 11 standard BBP etypes. These bundled defaults are used automatically.

**Custom Parameters:**  
For circuits with custom etypes not covered by the defaults, you'll need to prepare custom `simplified_parameters.json` and call the pipeline directly.

**NEURON Mechanisms:**  
The `single_compartment` algorithm requires NEURON with the circuit's MOD files compiled. If you see mechanism errors, run:
```bash
cd /path/to/circuit
nrnivmodl mod
```

**Production Workflow:**  
In production, the TaskManager will:
1. Create a `CircuitSimplificationSingleConfig` from the campaign config
2. Stage the circuit from entitycore
3. Run the simplification in a container with pre-compiled mechanisms
4. Register the output circuit with derivation links to the parent